# Approach 2 - E1: check once at the start, then check the agent's plan

Same benchmark (SimuHome QT4), same agent and scorer as approach 1. What changes is the gate:

| | approach 1 (`v1_gate`) | approach 2 |
|---|---|---|
| when the request is checked | inside every `schedule_workflow` call | once, before the agent's first step |
| contradictory request | 409; the agent retries other times | the agent is told at step 0 (`hint`), or the gate answers the user with no agent call (`answer`); a later call gets a final rejection |
| consistent request | always passes | the call's start time must fit the goal's Z3 window; the steps are dry-run on a copy of the home |
| episodes in parallel | 1 | `WORKERS` (each with its own simulator; one rate-limited key) |

**Runs on CPU** when a parse cache is attached (`parse_cache_v3.json` from `01_train_parser_v3`, or a v2
cache this notebook can build from the v2 adapter on a GPU session). Kaggle settings: Internet ON, secret
`NVIDIA_API_KEY`. Edit only the blocks marked EDIT. *Save Version -> Save & Run All* runs it unattended;
results land in `approach2_results.zip`.


## 1. Setup


In [ ]:
import os, sys
WORK = '/kaggle/working' if os.path.isdir('/kaggle/working') else os.getcwd()
os.chdir(WORK); PROJECT_ROOT = WORK
if not os.path.isdir('SimuHome'):
    !git clone --depth 1 https://github.com/holi-lab/SimuHome.git
# pyproject.toml asks for Python >= 3.13; the code runs on 3.10/3.11, so skip the gate.
!pip install -e ./SimuHome -q --ignore-requires-python
!pip -q install z3-solver
!pip uninstall -y -q torchao   # Kaggle's torchao 0.10 breaks peft when the parser adapter is loaded on a GPU
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'SimuHome'))
sys.path.insert(0, PROJECT_ROOT)
import glob
_n = len(glob.glob(os.path.join(PROJECT_ROOT, 'SimuHome/data/benchmark/qt4-*.json')))
assert _n == 300, f'expected 300 QT4 episodes, found {_n}: the SimuHome clone is missing or incomplete (delete SimuHome/ and rerun)'


In [ ]:
import os; os.makedirs('irlib', exist_ok=True); open('irlib/__init__.py', 'a').close()


In [ ]:
%%writefile irlib/ir_schema.py
"""Typed intermediate representation (IR v3) for SimuHome QT4 scheduling requests.

An IR describes one `schedule_workflow`-shaped user request as a set of goals.
Each goal is a set of target device assertions gated by one or more *time
readings* (ways the natural-language query says when the action happens).
A feasible request has one consistent reading per goal; an infeasible one states
two readings that resolve to different times for the same action (SimuHome's
"Contradiction Blindness" failure mode).

Changes from v2 (approach 1):
  - Anchor kind "goal": a reading measured from an EARLIER goal of the same
    request ("8 minutes after the previous action"). v2 had no such anchor and
    labelled these readings as "now"-anchored, which is harmless for a pure
    contradiction check but wrong for checking the agent's start_time.
  - Relation "by": a deadline (the state must hold no later than t). SimuHome
    marks these goals with tolerance_ticks == 0 and relation "before" (e.g. QT4-3
    "start the dryer now ... pause it when the washer finishes": the dryer only
    has to be running before the washer ends). v2 treated them as a point.
"""

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Literal, Optional

Relation = Literal["after", "before", "at", "by"]
AnchorKind = Literal["now", "device_event", "goal"]

IR_VERSION = 3


@dataclass
class Anchor:
    """What a time offset is measured from."""

    kind: AnchorKind = "now"
    room_id: Optional[str] = None
    device_id: Optional[str] = None
    device_type: Optional[str] = None
    goal_ref: Optional[int] = None  # kind == "goal": index of the earlier goal

    def to_dict(self) -> dict[str, Any]:
        return {
            "kind": self.kind,
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
            "goal_ref": self.goal_ref,
        }


@dataclass
class TimeReading:
    """One way the query expresses *when* an action should occur.

    All times are minutes since the episode's `base_time` (the moment the user
    spoke). A second reading on the same goal that disagrees with the first is
    what makes a request infeasible.
    """

    relation: Relation
    anchor: Anchor
    offset_minutes: Optional[float] = None  # relative reading
    absolute_time: Optional[str] = None  # "HH:MM[:SS]", clock-time reading
    # device_event anchors only: when the anchor device finishes (minutes since
    # base_time). Never in the training target; filled from live simulator state.
    resolved_at_minutes: Optional[float] = None
    tolerance_minutes: float = 0.0

    def to_dict(self) -> dict[str, Any]:
        return {
            "relation": self.relation,
            "anchor": self.anchor.to_dict(),
            "offset_minutes": self.offset_minutes,
            "absolute_time": self.absolute_time,
            "resolved_at_minutes": self.resolved_at_minutes,
            "tolerance_minutes": self.tolerance_minutes,
        }


@dataclass
class Assertion:
    attribute: str
    value: Any
    value_type: Optional[str] = None
    description: Optional[str] = None

    def to_dict(self) -> dict[str, Any]:
        return {
            "attribute": self.attribute,
            "value": self.value,
            "value_type": self.value_type,
            "description": self.description,
        }


@dataclass
class Target:
    room_id: str
    device_id: str
    device_type: Optional[str]
    asserts: list[Assertion] = field(default_factory=list)

    def to_dict(self) -> dict[str, Any]:
        return {
            "room_id": self.room_id,
            "device_id": self.device_id,
            "device_type": self.device_type,
            "asserts": [a.to_dict() for a in self.asserts],
        }


@dataclass
class Goal:
    """One scheduled effect: a set of device assertions gated by time reading(s)."""

    goal_id: int
    readings: list[TimeReading]
    targets: list[Target]
    # Set when the conflict is a duration/resource issue rather than a stated-time
    # mismatch (e.g. "finish the washer exactly when the dishwasher finishes" but the
    # washer cycle is longer). Not checkable without a device duration model.
    requires_duration_model: bool = False

    def to_dict(self) -> dict[str, Any]:
        return {
            "goal_id": self.goal_id,
            "readings": [r.to_dict() for r in self.readings],
            "targets": [t.to_dict() for t in self.targets],
            "requires_duration_model": self.requires_duration_model,
        }


@dataclass
class IntentIR:
    query: str
    query_type: str
    case: Literal["feasible", "infeasible", "unknown"]
    base_time: str  # "YYYY-MM-DD HH:MM:SS"
    goals: list[Goal] = field(default_factory=list)
    conflict_type: Optional[str] = None  # ground-truth label, training-time only

    def to_dict(self) -> dict[str, Any]:
        return {
            "query": self.query,
            "query_type": self.query_type,
            "case": self.case,
            "base_time": self.base_time,
            "goals": [g.to_dict() for g in self.goals],
            "conflict_type": self.conflict_type,
        }


def ir_from_dict(d: dict) -> IntentIR:
    """Inverse of IntentIR.to_dict() (used for the debug ground-truth IRs)."""
    goals = []
    for g in d["goals"]:
        readings = [
            TimeReading(anchor=Anchor(**r["anchor"]), **{k: v for k, v in r.items() if k != "anchor"})
            for r in g["readings"]
        ]
        targets = [
            Target(
                room_id=t["room_id"],
                device_id=t["device_id"],
                device_type=t.get("device_type"),
                asserts=[Assertion(**a) for a in t.get("asserts", [])],
            )
            for t in g["targets"]
        ]
        goals.append(
            Goal(
                goal_id=g["goal_id"],
                readings=readings,
                targets=targets,
                requires_duration_model=g.get("requires_duration_model", False),
            )
        )
    return IntentIR(
        query=d["query"],
        query_type=d["query_type"],
        case=d["case"],
        base_time=d["base_time"],
        goals=goals,
        conflict_type=d.get("conflict_type"),
    )


# ---------------------------------------------------------------------------
# Compact wire format (what the SLM is trained to emit)
# ---------------------------------------------------------------------------
# Keeps only what the query text determines: per goal, the time readings and the
# targets. Z3 decides feasibility; the model never says it. v3 adds `goal_ref`
# for goal anchors and the "by" relation; everything else is unchanged from v2,
# so a v2 parser's output still loads (it just never produces goal anchors).

DEFAULT_TOLERANCE_MINUTES = 0.5


def compact_ir_dict(ir_dict: dict, *, keep_resolved: bool = False) -> dict:
    """Full IR dict -> compact dict. Clock times are cut to HH:MM (the query
    states minutes, never the simulator's seconds)."""
    goals = []
    for g in ir_dict["goals"]:
        readings = []
        for r in g["readings"]:
            anchor = {"kind": r["anchor"]["kind"]}
            if r["anchor"].get("device_id"):
                anchor["device_id"] = r["anchor"]["device_id"]
            if r["anchor"].get("room_id"):
                anchor["room_id"] = r["anchor"]["room_id"]
            if r["anchor"].get("goal_ref") is not None:
                anchor["goal_ref"] = r["anchor"]["goal_ref"]
            out = {"relation": r["relation"], "anchor": anchor}
            if r.get("offset_minutes") is not None:
                out["offset_minutes"] = r["offset_minutes"]
            if r.get("absolute_time") is not None:
                out["absolute_time"] = ":".join(r["absolute_time"].split(":")[:2])
            if keep_resolved and r.get("resolved_at_minutes") is not None:
                out["resolved_at_minutes"] = r["resolved_at_minutes"]
            readings.append(out)
        targets = [
            {
                "device_id": t["device_id"],
                "asserts": [{"attribute": a["attribute"], "value": a["value"]} for a in t.get("asserts", [])],
            }
            for t in g["targets"]
        ]
        goals.append({"readings": readings, "targets": targets})
    return {"goals": goals}


def ir_from_compact(compact: dict, query: str, base_time: str) -> IntentIR:
    """Compact dict (+ the live query and the episode base_time) -> IntentIR."""
    goals = []
    for i, g in enumerate(compact["goals"]):
        readings = []
        for r in g["readings"]:
            a = r.get("anchor") or {}
            readings.append(
                TimeReading(
                    relation=r.get("relation", "after"),
                    anchor=Anchor(
                        kind=a.get("kind", "now"),
                        room_id=a.get("room_id"),
                        device_id=a.get("device_id"),
                        goal_ref=a.get("goal_ref"),
                    ),
                    offset_minutes=r.get("offset_minutes"),
                    absolute_time=r.get("absolute_time"),
                    resolved_at_minutes=r.get("resolved_at_minutes"),
                    tolerance_minutes=r.get("tolerance_minutes", DEFAULT_TOLERANCE_MINUTES),
                )
            )
        targets = [
            Target(
                room_id=t.get("room_id", ""),
                device_id=t["device_id"],
                device_type=None,
                asserts=[Assertion(attribute=a["attribute"], value=a["value"]) for a in t.get("asserts", [])],
            )
            for t in g.get("targets", [])
        ]
        goals.append(Goal(goal_id=i, readings=readings, targets=targets))
    return IntentIR(query=query, query_type="qt4", case="unknown", base_time=base_time, goals=goals)


def uses_goal_anchors(compact: dict) -> bool:
    """True if a compact IR contains any v3 goal anchor (i.e. came from a v3 parser
    or from v3 labels). Used to decide whether start_time checks are trustworthy."""
    return any(
        (r.get("anchor") or {}).get("kind") == "goal"
        for g in compact.get("goals", [])
        for r in g.get("readings", [])
    )


In [ ]:
%%writefile irlib/constraint_encoder.py
"""Component 2: IR -> Z3 constraints over real-valued time; SAT/UNSAT and time windows.

Time is one Real variable per goal: minutes since the episode's `base_time` (the
moment the user spoke, "now" = 0). Each TimeReading adds one constraint.

Changes from approach 1 (which solved every goal separately):
  - All goals are encoded in ONE solver, so readings can refer to each other:
    goal anchors ("8 min after the previous action") and a device anchor shared
    by several goals ("10 and 20 min after the dishwasher finishes" must agree on
    when the dishwasher finishes).
  - "by" readings are deadlines (t <= target + tol), not points.
  - When the request is consistent, `goal_windows` returns the interval [lo, hi]
    of start times each goal allows. The gate checks the agent's
    `schedule_workflow(start_time=...)` against these windows. This is the part a
    one-line Python comparison cannot do once goals chain off each other.

CPU-only; no GPU, simulator or SLM needed to run or test this module.
"""

from __future__ import annotations

import functools
import threading
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from typing import Optional

import z3

from .ir_schema import Goal, IntentIR, TimeReading

# Z3's default context is not thread-safe: two episodes checking at the same time
# (the parallel runner) crash it with "unreachable" / "invalid argument". Every
# public entry point holds this lock; a check takes milliseconds, so it costs nothing.
_Z3_LOCK = threading.RLock()


def _z3_serialized(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        with _Z3_LOCK:
            return fn(*args, **kwargs)
    return wrapper


def _parse_hms_to_minutes_of_day(hms: str) -> float:
    parts = [int(x) for x in hms.split(":")]
    h, m = parts[0], parts[1]
    s = parts[2] if len(parts) > 2 else 0
    return h * 60.0 + m + s / 60.0


def _is_minute_granular(hms: str) -> bool:
    return len(hms.split(":")) < 3


def _base_time_minutes_of_day(base_time: str) -> float:
    return _parse_hms_to_minutes_of_day(base_time.split(" ", 1)[1])


def minutes_to_clock(base_time: str, minutes: float) -> str:
    """Minutes since base_time -> "HH:MM:SS" wall-clock string."""
    base = datetime.strptime(base_time, "%Y-%m-%d %H:%M:%S")
    return (base + timedelta(minutes=float(minutes))).strftime("%H:%M:%S")


def clock_to_minutes(base_time: str, timestamp: str) -> float:
    """"YYYY-MM-DD HH:MM:SS" -> minutes since base_time."""
    base = datetime.strptime(base_time, "%Y-%m-%d %H:%M:%S")
    return (datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S") - base).total_seconds() / 60.0


def reading_to_absolute_minutes(reading: TimeReading, base_time: str) -> Optional[float]:
    """Minutes since base_time for a self-contained clock-time reading, else None."""
    if reading.absolute_time is not None and reading.anchor.kind == "now":
        # Clock times are same-day in SimuHome QT4 episodes; no midnight wrap handling.
        return _parse_hms_to_minutes_of_day(reading.absolute_time) - _base_time_minutes_of_day(base_time)
    return None


@dataclass
class GoalCheckResult:
    goal_id: int
    sat: bool
    reason: Optional[str] = None  # populated on UNSAT (or "skipped: ...")
    unsat_core: Optional[list[str]] = None
    resolved_minutes: Optional[float] = None  # one consistent time, on SAT


@dataclass
class GoalWindow:
    """Start times (minutes since base_time) a goal allows. None = unbounded."""

    goal_id: int
    lo: Optional[float]
    hi: Optional[float]
    deadline: bool = False  # only an upper bound was stated ("by")

    def contains(self, minutes: float, slack: float = 0.0) -> bool:
        if self.lo is not None and minutes < self.lo - slack:
            return False
        if self.hi is not None and minutes > self.hi + slack:
            return False
        return True


@dataclass
class FeasibilityResult:
    feasible: bool
    goal_results: list[GoalCheckResult]
    base_time: Optional[str] = None
    core_descriptions: list[str] = field(default_factory=list)

    def explanation(self) -> str:
        bad = [g for g in self.goal_results if not g.sat]
        if not bad:
            return "All scheduling constraints are consistent."
        lines = [f"Goal {g.goal_id}: {g.reason}" for g in bad]
        return "Infeasible schedule - " + "; ".join(lines)


def _describe_reading(reading: TimeReading, base_time: Optional[str], value: Optional[float]) -> str:
    when = f" (= {minutes_to_clock(base_time, value)})" if (base_time and value is not None) else ""
    if reading.absolute_time is not None:
        return f"stated clock time {reading.absolute_time}"
    if reading.offset_minutes is None:
        return "unspecified time reading"
    if reading.anchor.kind == "now":
        anchor_desc = "now"
    elif reading.anchor.kind == "goal":
        anchor_desc = f"goal {reading.anchor.goal_ref}"
    else:
        anchor_desc = f"{reading.anchor.device_id} finishing"
    rel = "before" if reading.relation == "by" else reading.relation
    prefix = "no later than " if reading.relation == "by" else ""
    return f"{prefix}{reading.offset_minutes:g} min {rel} {anchor_desc}{when}"


class _Encoding:
    """Builds the joint constraint system once; reused for the check and the windows."""

    def __init__(self, ir: IntentIR):
        self.ir = ir
        self.t = {g.goal_id: z3.Real(f"t_goal_{g.goal_id}") for g in ir.goals}
        self.anchors: dict[str, z3.ArithRef] = {}
        self.pins: list[tuple[str, z3.BoolRef]] = []  # known anchor times (not "readings")
        self.readings: list[tuple[str, z3.BoolRef, int, TimeReading, Optional[float]]] = []
        self.skipped: dict[int, str] = {}
        for goal in ir.goals:
            self._encode_goal(goal)

    def _anchor_var(self, device_id: str, resolved: Optional[float], label: str) -> z3.ArithRef:
        if device_id not in self.anchors:
            self.anchors[device_id] = z3.Real(f"anchor_{device_id}")
        var = self.anchors[device_id]
        if resolved is not None:
            self.pins.append((f"{label}_anchor_known", var == z3.RealVal(resolved)))
        return var

    def _encode_goal(self, goal: Goal) -> None:
        if goal.requires_duration_model:
            self.skipped[goal.goal_id] = "skipped: requires a device duration model (not checked)"
            return
        t = self.t[goal.goal_id]
        for i, reading in enumerate(goal.readings):
            label = f"g{goal.goal_id}_r{i}"
            extra = 1.0 if (reading.absolute_time and _is_minute_granular(reading.absolute_time)) else 0.0
            # An "HH:MM" clock time is only known to the minute, so widen by one minute.
            tol = z3.RealVal(reading.tolerance_minutes + extra)
            sign = -1 if reading.relation in ("before", "by") else 1
            nominal: Optional[float] = None  # the reading's own time, when known, for explanations

            if reading.anchor.kind == "device_event":
                anchor_var = self._anchor_var(reading.anchor.device_id or "?", reading.resolved_at_minutes, label)
                if reading.offset_minutes is None:
                    continue
                offset = 0.0 if reading.relation == "at" else sign * reading.offset_minutes
                target = anchor_var + z3.RealVal(offset)
                if reading.resolved_at_minutes is not None:
                    nominal = reading.resolved_at_minutes + offset
            elif reading.anchor.kind == "goal":
                ref = reading.anchor.goal_ref
                if ref is None or ref not in self.t or ref == goal.goal_id or reading.offset_minutes is None:
                    continue  # malformed reference (e.g. a parser slip): no constraint rather than a wrong one
                target = self.t[ref] + z3.RealVal(sign * reading.offset_minutes)
            else:
                resolved = reading_to_absolute_minutes(reading, self.ir.base_time)
                if resolved is not None:
                    target = z3.RealVal(resolved)
                    nominal = resolved
                elif reading.offset_minutes is not None:
                    target = z3.RealVal(sign * reading.offset_minutes)
                    nominal = sign * reading.offset_minutes
                else:
                    continue

            if reading.relation == "by":
                constraint = t <= target + tol
            else:
                constraint = z3.And(t >= target - tol, t <= target + tol)
            self.readings.append((label, constraint, goal.goal_id, reading, nominal))

    def solver(self, track: bool) -> z3.Solver:
        s = z3.Solver()
        for label, c in self.pins:
            if track:
                s.assert_and_track(c, z3.Bool(label))
            else:
                s.add(c)
        for label, c, *_ in self.readings:
            if track:
                s.assert_and_track(c, z3.Bool(label))
            else:
                s.add(c)
        return s


@_z3_serialized
def check_feasibility(ir: IntentIR) -> FeasibilityResult:
    enc = _Encoding(ir)
    solver = enc.solver(track=True)
    if solver.check() == z3.sat:
        model = solver.model()
        results = []
        for g in ir.goals:
            if g.goal_id in enc.skipped:
                results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, reason=enc.skipped[g.goal_id]))
                continue
            val = model.eval(enc.t[g.goal_id], model_completion=True)
            results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, resolved_minutes=float(val.as_fraction())))
        return FeasibilityResult(feasible=True, goal_results=results, base_time=ir.base_time)

    core = {str(c) for c in solver.unsat_core()}
    by_label = {label: (gid, r, nominal) for label, _, gid, r, nominal in enc.readings}
    bad_goals: dict[int, list[str]] = {}
    descriptions = []
    # "*_anchor_known" pins only fix a symbolic anchor to a runtime-known value;
    # they are not a statement in the request, so they are not named.
    for label in sorted(core):
        if label in by_label:
            gid, r, nominal = by_label[label]
            d = _describe_reading(r, ir.base_time, nominal)
            bad_goals.setdefault(gid, []).append(d)
            descriptions.append(f"goal {gid}: {d}")
    results = []
    for g in ir.goals:
        if g.goal_id in bad_goals:
            results.append(
                GoalCheckResult(
                    goal_id=g.goal_id,
                    sat=False,
                    reason=" vs. ".join(bad_goals[g.goal_id]),
                    unsat_core=sorted(core),
                )
            )
        else:
            results.append(GoalCheckResult(goal_id=g.goal_id, sat=True, reason=enc.skipped.get(g.goal_id)))
    if not bad_goals:  # core made only of pins (should not happen); still report UNSAT honestly
        results[0] = GoalCheckResult(goal_id=results[0].goal_id, sat=False, reason="unsatisfiable", unsat_core=sorted(core))
    return FeasibilityResult(feasible=False, goal_results=results, base_time=ir.base_time, core_descriptions=descriptions)


def _bound(enc: _Encoding, var: z3.ArithRef, maximize: bool) -> Optional[float]:
    opt = z3.Optimize()
    for _, c in enc.pins:
        opt.add(c)
    for _, c, *_ in enc.readings:
        opt.add(c)
    h = opt.maximize(var) if maximize else opt.minimize(var)
    if opt.check() != z3.sat:
        return None
    value = opt.upper(h) if maximize else opt.lower(h)
    # Z3 hands back an integer or rational numeral when the bound is finite, and a
    # non-numeral term (oo, epsilon) when the goal is unbounded in that direction.
    if z3.is_int_value(value):
        return float(value.as_long())
    if z3.is_rational_value(value):
        return float(value.as_fraction())
    return None


@_z3_serialized
def goal_windows(ir: IntentIR) -> dict[int, GoalWindow]:
    """For a consistent IR: the [lo, hi] start-time interval of every goal.

    Goals whose readings are all unresolved (e.g. the anchor device is idle, so its
    finish time is unknown) come back unbounded, and the gate does not check them.
    """
    enc = _Encoding(ir)
    if enc.solver(track=False).check() != z3.sat:
        return {}
    windows = {}
    for g in ir.goals:
        if g.goal_id in enc.skipped:
            windows[g.goal_id] = GoalWindow(goal_id=g.goal_id, lo=None, hi=None)
            continue
        t = enc.t[g.goal_id]
        lo, hi = _bound(enc, t, maximize=False), _bound(enc, t, maximize=True)
        deadline = all(r.relation == "by" for r in g.readings) and bool(g.readings)
        windows[g.goal_id] = GoalWindow(goal_id=g.goal_id, lo=lo, hi=hi, deadline=deadline)
    return windows


In [ ]:
%%writefile irlib/extract_ir_from_episode.py
"""Component: deterministic (query, IR) pair extraction from SimuHome episodes.

No LLM calls, no manual labeling. SimuHome benchmark/generated episodes
already carry, in `eval.goals` (ground-truth required behavior) and
`temporal_conflict` (ground-truth contradiction, infeasible episodes only),
every field needed to derive the IR mechanically. See handoff Sec 5.3.

Two IRs are produced per episode:
  - `ir_for_training`: what the SLM must learn to produce from the query
    text alone. Device-anchored readings never carry a numeric resolved
    time here, because the query text does not state one — the live
    simulator state does, and only the runtime agent can fetch it.
  - `debug_ground_truth`: the same IR with anchor times filled in from the
    simulator-computed fields, for exercising the Z3 encoder end-to-end
    without needing a live simulator.

IR v3 (approach 2) differs from v2 in two labelling rules, both read off
`eval.goals[].when`:
  - a goal with no device anchor but `reference_minutes > 0` is chained to the
    earlier goal whose `at_minutes` equals that reference ("8 minutes after the
    previous action") -> Anchor(kind="goal", goal_ref=j);
  - `relation == "before"` with `tolerance_ticks == 0` is a deadline (the state
    must hold by then, e.g. QT4-3 "start the dryer now") -> relation "by".
"""

from __future__ import annotations

import glob
import json
import os
from typing import Any, Optional

from .ir_schema import Anchor, Assertion, Goal, IntentIR, Target, TimeReading, compact_ir_dict

# Conflict types where the contradiction is a stated-time mismatch (checkable
# from two TimeReadings alone). See handoff Sec 6: duration/resource
# feasibility conflicts (e.g. "finish washer exactly when dishwasher ends"
# but the washer cycle itself is longer) need a device duration model the
# Z3 encoder does not have yet — flagged, not silently mis-scored.
_DURATION_MODEL_CONFLICT_TYPES = {"completion_vs_pause"}


def _tolerance_minutes(when: dict[str, Any], tick_interval: float) -> float:
    return float(when.get("tolerance_ticks", 0)) * tick_interval / 60.0


def _chained_goal_ref(
    goal_id: int, when: dict[str, Any], raw_goals: list[dict[str, Any]], *, any_direction: bool = False
) -> Optional[int]:
    """Index of the goal this one is measured from, or None.

    SimuHome writes "8 minutes after the previous action" as relation "after",
    offset 8, with `reference_minutes` = the previous goal's `at_minutes` and no
    device anchor; the latest earlier goal with that time is the reference.
    Deadline goals (QT4-3, `any_direction=True`) reference the OTHER goal's time
    ("the dryer must be running 1 min before it is paused"), which can be a later goal.
    """
    ref = when.get("reference_minutes")
    if not ref:
        return None
    for j in range(goal_id - 1, -1, -1):
        if raw_goals[j]["when"].get("at_minutes") == ref:
            return j
    if any_direction:
        for j in range(goal_id + 1, len(raw_goals)):
            if raw_goals[j]["when"].get("at_minutes") == ref:
                return j
    return None


def _device_anchor(raw_anchor: dict[str, Any]) -> Anchor:
    return Anchor(
        kind="device_event",
        room_id=raw_anchor.get("room_id"),
        device_id=raw_anchor.get("device_id"),
        device_type=raw_anchor.get("device_type"),
    )


def _goal_from_eval_goal(
    goal_id: int,
    raw_goal: dict[str, Any],
    tick_interval: float,
    raw_goals: Optional[list[dict[str, Any]]] = None,
    for_training: bool = True,
) -> Goal:
    when = raw_goal["when"]
    raw_anchor = raw_goal.get("anchor") or {}

    relation = when.get("relation", "after")
    deadline = relation == "before" and int(when.get("tolerance_ticks", 0)) == 0
    if deadline:
        relation = "by"  # deadline: the state must hold no later than anchor - offset

    goal_ref = _chained_goal_ref(goal_id, when, raw_goals or [], any_direction=deadline)
    if deadline and goal_ref is not None:
        # Measured from the other goal, even when eval.goals labels a device anchor:
        # its reference_minutes is that goal's time, not the device's finish.
        anchor = Anchor(kind="goal", goal_ref=goal_ref)
    elif raw_anchor.get("device_id"):
        anchor = _device_anchor(raw_anchor)
    elif goal_ref is not None:
        anchor = Anchor(kind="goal", goal_ref=goal_ref)
    else:
        anchor = Anchor(kind="now")

    reading = TimeReading(
        relation=relation,
        anchor=anchor,
        offset_minutes=float(when.get("offset_minutes", 0.0)),
        tolerance_minutes=_tolerance_minutes(when, tick_interval),
    )
    if anchor.kind == "device_event" and not for_training and when.get("reference_tick") is not None:
        # Debug IR only: when the anchor device finishes, as the runtime reads it from
        # the simulator (CountdownTime). The training IR never contains it. Ticks, not
        # reference_minutes, which SimuHome rounds to whole minutes.
        reading.resolved_at_minutes = float(when["reference_tick"]) * tick_interval / 60.0

    targets = [
        Target(
            room_id=t["room_id"],
            device_id=t["device_id"],
            device_type=t.get("device_type"),
            asserts=[
                Assertion(
                    attribute=a["attribute"],
                    value=a["value"],
                    value_type=a.get("value_type"),
                    description=a.get("description"),
                )
                for a in t.get("asserts", [])
            ],
        )
        for t in raw_goal.get("targets", [])
    ]

    return Goal(goal_id=goal_id, readings=[reading], targets=targets)


def _inject_conflicting_reading(
    goal: Goal, temporal_conflict: dict[str, Any], for_training: bool, raw_anchor: Optional[dict[str, Any]] = None
) -> Goal:
    """Add the second, contradicting TimeReading to the primary goal.

    Mutates and returns `goal`. If the conflict is duration/resource-based
    (no comparable second time reading exists), flags the goal instead.
    """
    conflict_type = temporal_conflict.get("type")

    if conflict_type in _DURATION_MODEL_CONFLICT_TYPES or "conflict_at" not in temporal_conflict:
        goal.requires_duration_model = True
        return goal

    anchor_end_at = temporal_conflict.get("anchor_end_at")
    expected_at = temporal_conflict.get("expected_at")
    conflict_time = temporal_conflict.get("conflict_time")
    relation = temporal_conflict.get("relation", "after")

    if anchor_end_at is not None:
        # Dependency-anchored primary reading (e.g. "20 min after dishwasher finishes").
        base_offset = temporal_conflict.get(
            "offset_minutes", temporal_conflict.get("delay_minutes", 0.0)
        )
        offset = float(base_offset)
        # The conflict is stated against the anchor DEVICE finishing, so use the
        # device from eval.goals even if v3 relabelled the goal as goal-anchored.
        anchor = _device_anchor(raw_anchor) if (raw_anchor or {}).get("device_id") else goal.readings[0].anchor
        primary_reading = TimeReading(
            relation=relation,
            anchor=anchor,
            offset_minutes=offset,
            tolerance_minutes=goal.readings[0].tolerance_minutes,
            # resolved_at_minutes here is the ANCHOR EVENT's own resolved
            # time (e.g. when the dishwasher finishes), not the target time
            # — the Z3 encoder derives target = anchor +/- offset itself.
            resolved_at_minutes=None if for_training else float(anchor_end_at),
        )
        goal.readings[0] = primary_reading
    elif expected_at is not None:
        # Now-anchored primary reading (e.g. "13 minutes from now").
        goal.readings[0].offset_minutes = float(expected_at)
        goal.readings[0].anchor = Anchor(kind="now")

    if conflict_time is not None:
        goal.readings.append(
            TimeReading(
                relation="at",
                anchor=Anchor(kind="now"),
                absolute_time=conflict_time.split(" ")[-1] if " " in conflict_time else conflict_time,
                tolerance_minutes=goal.readings[0].tolerance_minutes,
            )
        )

    return goal


def extract_episode(episode: dict[str, Any], *, for_training: bool = True) -> IntentIR:
    """Build an IntentIR from one raw SimuHome benchmark/generated episode dict."""
    meta = episode["meta"]
    tick_interval = float(episode["initial_home_config"]["tick_interval"])
    base_time = episode["initial_home_config"]["base_time"]
    case = meta.get("case", "unknown")

    raw_goals = episode["eval"]["goals"]
    goals = [
        _goal_from_eval_goal(i, rg, tick_interval, raw_goals, for_training) for i, rg in enumerate(raw_goals)
    ]

    conflict_type = None
    if case == "infeasible" and "temporal_conflict" in episode:
        temporal_conflict = episode["temporal_conflict"]
        conflict_type = temporal_conflict.get("type")
        first = goals[0].readings[0] if goals and goals[0].readings else None
        if (
            first is not None
            and first.relation == "by"
            and first.anchor.kind == "goal"
            and conflict_type not in _DURATION_MODEL_CONFLICT_TYPES
            and temporal_conflict.get("conflict_time")
        ):
            # QT4-3: every goal keeps exactly the reading its FEASIBLE twin has; the stated
            # clock time is added to the goal it is about. v3.0 instead replaced goal 0's
            # reading, so feasible and infeasible QT4-3 labels differed in more than the clock
            # time, and the parser copied the infeasible template (with an invented clock time)
            # onto feasible requests (held-out seeds 55, 75).
            #   absolute_time_mismatch: "start at 6:14 PM ... pause when X finishes" -> the start (goal 0)
            #   delay_time_mismatch:    "start 14 min after X finishes, at 11:40"  -> the delayed start
            #   impossible_early_end:   "finish together before 2:35 PM"           -> the finish
            if conflict_type == "absolute_time_mismatch":
                target = goals[0]
            else:
                target = next((g for g in goals if g.readings and g.readings[0].anchor.kind == "device_event"), goals[0])
            clock = temporal_conflict["conflict_time"]
            target.readings.append(
                TimeReading(
                    relation="at",
                    anchor=Anchor(kind="now"),
                    absolute_time=clock.split(" ")[-1],
                    tolerance_minutes=target.readings[0].tolerance_minutes,
                )
            )
        elif goals:
            goals[0] = _inject_conflicting_reading(
                goals[0], temporal_conflict, for_training, raw_goals[0].get("anchor")
            )

    return IntentIR(
        query=episode["query"],
        query_type=meta.get("query_type", "unknown"),
        case=case,
        base_time=base_time,
        goals=goals,
        conflict_type=conflict_type,
    )


def extract_pair(episode: dict[str, Any]) -> dict[str, Any]:
    """Return {"query", "ir_for_training", "debug_ground_truth", "meta"} for one episode."""
    training_ir = extract_episode(episode, for_training=True)
    debug_ir = extract_episode(episode, for_training=False)
    return {
        "query": episode["query"],
        "ir_for_training": compact_ir_dict(training_ir.to_dict()),
        "debug_ground_truth": debug_ir.to_dict(),
        "meta": episode["meta"],
    }


def extract_dataset(benchmark_dir: str, pattern: str = "qt4-*_seed_*.json") -> list[dict[str, Any]]:
    """Extract (query, IR) pairs for every QT4 episode file in `benchmark_dir`."""
    # Benchmark files sit directly in the dir; generator output is <run>/episodes/*.json.
    paths = sorted(
        glob.glob(os.path.join(benchmark_dir, "qt4-*.json"))
        + glob.glob(os.path.join(benchmark_dir, "*", "episodes", "qt4-*.json"))
    )
    pairs = []
    errors = []
    for path in paths:
        try:
            with open(path, "r", encoding="utf-8") as f:
                episode = json.load(f)
            pair = extract_pair(episode)
            pair["source_file"] = os.path.basename(path)
            pairs.append(pair)
        except Exception as e:  # keep going; report at the end
            errors.append((path, str(e)))
    if errors:
        print(f"[extract_dataset] {len(errors)} file(s) failed to extract:")
        for path, err in errors[:20]:
            print(f"  {os.path.basename(path)}: {err}")
    return pairs


if __name__ == "__main__":
    import sys

    bench_dir = sys.argv[1] if len(sys.argv) > 1 else "SimuHome/data/benchmark"
    out_path = sys.argv[2] if len(sys.argv) > 2 else "data/qt4_ir_pairs.jsonl"
    pairs = extract_dataset(bench_dir)
    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    with open(out_path, "w", encoding="utf-8") as f:
        for pair in pairs:
            f.write(json.dumps(pair, ensure_ascii=False) + "\n")
    print(f"Extracted {len(pairs)} pairs -> {out_path}")


In [ ]:
%%writefile irlib/grounding.py
"""Grounding filter: drop clock times the parser wrote that the user never said.

The v3 parser sometimes adds an absolute clock reading that is not in the request
(e.g. "at 16:00" for "start the dishwasher now"). Z3 then finds a genuine clash
with the other reading and calls a POSSIBLE request impossible. A clock time is
something the user states explicitly, so it can be checked against the sentence
deterministically: if no time in the text matches it, the reading is removed
before Z3 sees it. Relative offsets are not filtered; a stated time plus/minus an offset in the same
IR also counts as stated (see ground_ir).

Times are recognised in digit form ("12:23 PM", "1:39 pm", "12 36 PM", "7 PM") and
word form ("six fifty six PM", "twelve thirty six PM", "seven o'clock").
"""

from __future__ import annotations

import copy
import re

_UNITS = {
    "zero": 0, "oh": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8,
    "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15,
    "sixteen": 16, "seventeen": 17, "eighteen": 18, "nineteen": 19,
}
_TENS = {"twenty": 20, "thirty": 30, "forty": 40, "fifty": 50}


def _with_meridiem(h: int, m: int, mer: str | None) -> set[tuple[int, int]]:
    if not (0 <= h <= 23 and 0 <= m <= 59):
        return set()
    if mer:
        mer = mer.replace(".", "").lower()
        if mer == "pm" and h < 12:
            h += 12
        elif mer == "am" and h == 12:
            h = 0
        return {(h, m)}
    out = {(h, m)}
    if h < 12:
        out.add((h + 12, m))  # no AM/PM given: either reading
    return out


def _words_to_numbers(text: str) -> str:
    """'six fifty six pm' -> '6 56 pm'; 'twelve thirty six' -> '12 36'."""
    tokens = re.findall(r"[a-z]+|\d+|[^a-z\d\s]", text.lower())
    out, i = [], 0
    while i < len(tokens):
        t = tokens[i]
        if t in _TENS:
            val = _TENS[t]
            if i + 1 < len(tokens) and tokens[i + 1] in _UNITS and 0 < _UNITS[tokens[i + 1]] < 10:
                val += _UNITS[tokens[i + 1]]
                i += 1
            out.append(str(val))
        elif t in _UNITS:
            val = _UNITS[t]
            if t == "oh" and i + 1 < len(tokens) and tokens[i + 1] in _UNITS:  # "ten oh three" -> "10 03"
                out.append(f"{_UNITS[tokens[i + 1]]:02d}")
                i += 2
                continue
            out.append(str(val))
        else:
            out.append(t)
        i += 1
    return " ".join(out)


def times_in_text(text: str) -> set[tuple[int, int]]:
    """Every (hour 0-23, minute) the text could be stating as a clock time."""
    found: set[tuple[int, int]] = set()
    for variant in (text.lower(), _words_to_numbers(text)):
        for h, m, mer in re.findall(r"\b(\d{1,2})\s*[:.\s]\s*(\d{2})\s*(a\.?m\.?|p\.?m\.?)?", variant):
            found |= _with_meridiem(int(h), int(m), mer or None)
        for h, mer in re.findall(r"\b(\d{1,2})\s*(a\.?m\.?|p\.?m\.?)", variant):
            found |= _with_meridiem(int(h), 0, mer)
        for h in re.findall(r"\b(\d{1,2})\s*o\s*'?\s*clock", variant):
            found |= _with_meridiem(int(h), 0, None)
    return found


def _closest_device(name: str, known: list[str], cutoff: float) -> str | None:
    """Closest real device id, preferring the same room (the id prefix before the device type)."""
    import difflib

    same_room = [k for k in known if k.split("_")[0] == name.split("_")[0]]
    for pool in (same_room, known):
        hit = difflib.get_close_matches(name, pool, n=1, cutoff=cutoff)
        if hit:
            return hit[0]
    return None


def repair_device_ids(compact: dict, known_ids: list[str], cutoff: float = 0.85) -> tuple[dict, int, int]:
    """Copy of `compact` whose device ids that do not exist in the house are replaced by the closest real
    one (e.g. the parser's 'utility_laundry_dryer_1' -> 'utility_room_laundry_dryer_1'). Uses only the
    house's device list, never the answer key. Returns (ir, n_repaired, n_unresolved)."""
    known = list(known_ids)
    known_set = set(known)
    out = copy.deepcopy(compact)
    repaired = unresolved = 0

    def fix(obj: dict) -> None:
        nonlocal repaired, unresolved
        dev = obj.get("device_id")
        if not dev or dev in known_set:
            return
        hit = _closest_device(dev, known, cutoff)
        if hit:
            obj["device_id"] = hit
            repaired += 1
        else:
            unresolved += 1

    for g in out.get("goals", []):
        for r in g.get("readings", []):
            if (r.get("anchor") or {}).get("kind") == "device_event":
                fix(r["anchor"])
        for t in g.get("targets", []):
            fix(t)
    return out, repaired, unresolved


def ground_ir(compact: dict, query: str) -> tuple[dict, int]:
    """Copy of `compact` without clock readings the query does not state. Returns (ir, n_dropped).

    A clock time counts as stated if it appears in the text, or if it is a stated time
    plus/minus an offset the same IR uses ("finishes around 4:42 PM ... start 19 minutes
    after" -> 17:01): the benchmark's own labels express such derived times this way.
    """
    stated = times_in_text(query)
    offsets = {int(round(r["offset_minutes"])) for g in compact.get("goals", []) for r in g.get("readings", [])
               if isinstance(r.get("offset_minutes"), (int, float))}
    derived = set()
    for h, m in stated:
        for o in offsets:
            for t in (h * 60 + m + o, h * 60 + m - o):
                derived.add(((t // 60) % 24, t % 60))
    stated = stated | derived
    out = copy.deepcopy(compact)
    dropped = 0
    for g in out.get("goals", []):
        kept = []
        for r in g.get("readings", []):
            at = r.get("absolute_time")
            if at:
                try:
                    h, m = (int(x) for x in at.split(":")[:2])
                except ValueError:
                    dropped += 1
                    continue
                if (h, m) not in stated:
                    dropped += 1
                    continue
            kept.append(r)
        g["readings"] = kept
    return out, dropped


In [ ]:
%%writefile irlib/gate.py
"""Component 3 (approach 2): the per-episode gate around `schedule_workflow`.

Approach 1 re-parsed the user's request on every `schedule_workflow` call, checked
only the request (never the agent's arguments) and sent back a 409 the agent
could not act on, so it kept retrying with other times. Approach 2 changes this:

  A. Pre-flight: parse once and run Z3 once, before the agent's first step. Device
     anchors are read straight from the simulator (no LLM call). If the request is
     contradictory, either tell the agent at step 0 ("hint") or answer the user
     directly without calling the agent at all ("answer").
  B. Final rejection: when the contradiction is in the request itself, the 409 says
     that no start_time can fix it and that the agent should call finish.
  C. Plan check, for consistent requests:
     C1 the call's start_time must fall inside the window Z3 computes for the goal
        whose devices the call touches;
     C2 the steps are dry-run on a copy of the home at that start_time
        (`shadow_sim`): steps that would fail are rejected with SimuHome's own error
        message; target attributes that would not reach the requested value are
        reported (warn) or rejected.
  Every rejection kind is capped per episode (`max_plan_rejections`); after that the
  gate fails open, so it can slow the agent down but never trap it in a loop.

Nothing here imports SimuHome: tool calls, the parser and the dry run are passed in,
so the logic is unit-testable without a simulator or GPU.
"""

from __future__ import annotations

import copy
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

from .constraint_encoder import (
    FeasibilityResult,
    GoalWindow,
    check_feasibility,
    clock_to_minutes,
    goal_windows,
    minutes_to_clock,
)
from .grounding import ground_ir, repair_device_ids
from .ir_schema import ir_from_compact, uses_goal_anchors

ToolFn = Callable[[str, dict], dict]


@dataclass
class GateConfig:
    preflight: str = "hint"  # "off" | "hint" | "answer"
    check_at_schedule: bool = True  # enforce the request-level verdict inside schedule_workflow
    terminal_rejection: bool = True  # B: say the contradiction cannot be fixed by another time
    time_check: str = "auto"  # "off" | "on" | "auto" (= on only for v3 IRs, which have goal anchors)
    dry_run: str = "reject"  # "off" | "warn" | "reject": steps that would fail when they run
    target_check: str = "warn"  # "off" | "warn" | "reject": target attributes not reaching the value
    sat_hints: bool = False  # also tell the agent the computed times when the request is consistent
    ground_clock_times: bool = True  # drop clock times the parser wrote but the user never said (grounding.py)
    ground_device_names: bool = True  # map device ids that do not exist in the house to the closest real one
    max_plan_rejections: int = 2  # per episode and per kind (time / dry_run / target); then fail open
    max_unsat_rejections: int = 3  # per episode, for an infeasible request
    slack_minutes: float = 0.25  # extra slack on every window (SimuHome scores within +/-30 s)
    # No dry run for start times further ahead than this (the copy must be fast-forwarded the whole
    # way: ~1 s per 10 simulated minutes). QT4 goals are at most ~116 min after the request.
    dry_run_horizon_minutes: float = 150.0
    ir_version: int = 2  # 3 when the parser was trained on v3 labels


# The arms of the experiment. `None` = SimuHome's original tool, no gate.
ARMS: dict[str, Optional[GateConfig]] = {
    "baseline": None,
    "selfcheck": None,  # + SELF_CHECK_PROMPT appended to the query (see the notebook)
    # approach 1 reproduced inside the same harness: check at schedule time only
    "v1_gate": GateConfig(preflight="off", terminal_rejection=False, time_check="off", dry_run="off", target_check="off",
                          ground_clock_times=False, ground_device_names=False),
    "pre_hint": GateConfig(preflight="hint", time_check="off", dry_run="off", target_check="off"),
    "pre_answer": GateConfig(preflight="answer", time_check="off", dry_run="off", target_check="off"),
    "full": GateConfig(preflight="hint"),
    "full_answer": GateConfig(preflight="answer"),
}


def _response(code: int, message: str, err_type: str, detail: str) -> dict:
    return {"status": {"code": code, "message": message}, "data": None, "error": {"type": err_type, "detail": detail}}


def _ok(resp: Any) -> bool:
    return isinstance(resp, dict) and (resp.get("status") or {}).get("code") == 200


def _tool_data(resp: Any) -> Any:
    return resp.get("data") if isinstance(resp, dict) else None


def explain_contradiction(result: FeasibilityResult) -> str:
    """Plain-English sentence naming the clashing statements (from the unsat core)."""
    bad = [g for g in result.goal_results if not g.sat]
    parts = [g.reason for g in bad if g.reason]
    return "; ".join(parts) if parts else "two parts of the request give the same action different times"


def user_answer(result: FeasibilityResult) -> str:
    """The reply sent to the user when the gate answers without the agent ("answer" pre-flight)."""
    return (
        "I can't schedule this as requested, because the timing contradicts itself: "
        f"{explain_contradiction(result)}. Both cannot be true, so I have not scheduled anything. "
        "Please tell me which time you want."
    )


def devices_in_steps(steps: list) -> list[str]:
    out = []
    for st in steps or []:
        args = st.get("args") if isinstance(st, dict) else None
        if isinstance(args, dict) and isinstance(args.get("device_id"), str) and args["device_id"] not in out:
            out.append(args["device_id"])
    return out


@dataclass
class GateStats:
    preflight: str = "not_run"  # not_run | sat | unsat | parse_fail | error
    preflight_seconds: float = 0.0
    clock_readings_dropped: int = 0  # parser clock times not found in the request (grounding filter)
    device_names_repaired: int = 0  # parser device ids not in the house, mapped to the closest real device
    device_names_unresolved: int = 0  # ... and ones with no close match (left as they are)
    schedule_calls: int = 0
    passed: int = 0
    unsat_rejections: int = 0
    time_rejections: int = 0
    dry_run_rejections: int = 0
    target_rejections: int = 0
    target_warnings: int = 0
    unmatched_calls: int = 0  # touched no device the IR knows: not checked
    fail_open: int = 0  # a cap was reached, or a check could not run
    dry_run_seconds: float = 0.0
    events: list = field(default_factory=list)

    def as_dict(self) -> dict:
        return {k: (v if k != "events" else list(v)) for k, v in self.__dict__.items()}


class EpisodeGate:
    """All gate state for ONE episode (one user request)."""

    def __init__(
        self,
        query: str,
        base_time: str,
        compact_ir: Optional[dict],
        cfg: GateConfig,
        call_tool: ToolFn,
        dry_run_fn: Optional[Callable[..., Any]] = None,
    ):
        self.query = query
        self.base_time = base_time  # when the user spoke = minute 0 of every reading
        self.compact_ir = compact_ir
        self.cfg = cfg
        self.call_tool = call_tool
        self.dry_run_fn = dry_run_fn
        self.stats = GateStats()
        self.result: Optional[FeasibilityResult] = None
        self.windows: dict[int, GoalWindow] = {}
        self.ir = None
        self.committed: list[dict] = []
        self._verdict_done = False

    # ------------------------------------------------------------------ verdict
    def _house_device_ids(self) -> list[str]:
        """Every device id in the house, from the simulator (one call; no answer key involved)."""
        snap = _tool_data(self.call_tool("get_home_state", {})) or {}
        return [d.get("device_id") for room in (snap.get("rooms") or {}).values() for d in (room.get("devices") or [])
                if d.get("device_id")]

    def _now_minutes(self) -> float:
        now = (_tool_data(self.call_tool("get_current_time", {})) or {}).get("now")
        return clock_to_minutes(self.base_time, now) if now else 0.0

    def _resolve_device_anchors(self, compact: dict) -> None:
        """Fill resolved_at_minutes for device anchors from the live simulator:
        finish time = now + CountdownTime. Idle devices stay unresolved (symbolic)."""
        needed = {
            r["anchor"]["device_id"]
            for g in compact["goals"]
            for r in g["readings"]
            if (r.get("anchor") or {}).get("kind") == "device_event" and r["anchor"].get("device_id")
            and r.get("resolved_at_minutes") is None
        }
        if not needed:
            return
        now_min = self._now_minutes()
        finish: dict[str, Optional[float]] = {}
        for dev in needed:
            resp = self.call_tool("get_attribute", {
                "device_id": dev, "endpoint_id": 1, "cluster_id": "OperationalState", "attribute_id": "CountdownTime",
            })
            data = _tool_data(resp)
            value = data.get("value") if isinstance(data, dict) else data
            ok = isinstance(value, (int, float)) and not isinstance(value, bool) and value > 0
            finish[dev] = now_min + float(value) / 60.0 if ok else None
        for g in compact["goals"]:
            for r in g["readings"]:
                a = r.get("anchor") or {}
                if a.get("kind") == "device_event" and r.get("resolved_at_minutes") is None:
                    r["resolved_at_minutes"] = finish.get(a.get("device_id"))

    def ensure_verdict(self) -> None:
        if self._verdict_done:
            return
        self._verdict_done = True
        t0 = time.time()
        if self.compact_ir is None:
            self.stats.preflight = "parse_fail"
            return
        try:
            compact = copy.deepcopy(self.compact_ir)
            if self.cfg.ground_clock_times:
                compact, self.stats.clock_readings_dropped = ground_ir(compact, self.query)
            if self.cfg.ground_device_names:
                known = self._house_device_ids()
                if known:
                    compact, self.stats.device_names_repaired, self.stats.device_names_unresolved =                         repair_device_ids(compact, known)
            self._resolve_device_anchors(compact)
            self.ir = ir_from_compact(compact, self.query, self.base_time)
            self.result = check_feasibility(self.ir)
            if self.result.feasible and self._time_check_on():
                self.windows = goal_windows(self.ir)
            self.stats.preflight = "sat" if self.result.feasible else "unsat"
        except Exception as e:  # a bug in the check must never break the episode: fail open
            self.result = None
            self.stats.preflight = "error"
            self.stats.events.append(("verdict_error", repr(e)[:200]))
        finally:
            self.stats.preflight_seconds += time.time() - t0

    def _time_check_on(self) -> bool:
        mode = self.cfg.time_check
        if mode == "auto":
            return self.cfg.ir_version >= 3 or bool(self.compact_ir and uses_goal_anchors(self.compact_ir))
        return mode == "on"

    @property
    def infeasible(self) -> bool:
        return self.result is not None and not self.result.feasible

    # ---------------------------------------------------------------- pre-flight
    def preflight_action(self) -> tuple[str, str]:
        """("none" | "hint" | "answer", text) to apply before the agent's first step."""
        if self.cfg.preflight == "off":
            return "none", ""
        self.ensure_verdict()
        if self.infeasible:
            if self.cfg.preflight == "answer":
                return "answer", user_answer(self.result)
            return "hint", (
                "\n\n[Note from the scheduling verifier, computed before you started: this request contains a "
                f"timing contradiction: {explain_contradiction(self.result)}. No start time can satisfy both "
                "statements. Do not call schedule_workflow; call finish and explain the conflict to the user.]"
            )
        if self.result is not None and self.cfg.sat_hints:
            parts = []
            for g in self.result.goal_results:
                if g.resolved_minutes is None:
                    continue
                devs = ", ".join(t.device_id for t in self.ir.goals[g.goal_id].targets) or "?"
                parts.append(f"{devs} at {minutes_to_clock(self.base_time, g.resolved_minutes)}")
            if parts:
                return "hint", "\n\n[Note from the scheduling verifier: the requested times work out to " + "; ".join(parts) + ".]"
        return "none", ""

    # ------------------------------------------------------------ schedule call
    def on_schedule(self, args: dict, real_tool: Callable[[dict], dict]) -> dict:
        self.stats.schedule_calls += 1
        if not self.cfg.check_at_schedule:
            return self._commit(args, real_tool)
        self.ensure_verdict()
        if self.result is None:  # parse failure or check error: fail open
            self.stats.fail_open += 1
            return self._commit(args, real_tool)

        if self.infeasible:
            if self.stats.unsat_rejections >= self.cfg.max_unsat_rejections:
                self.stats.fail_open += 1
                return self._commit(args, real_tool)
            self.stats.unsat_rejections += 1
            why = explain_contradiction(self.result)
            if self.cfg.terminal_rejection:
                detail = (
                    f"The request itself is contradictory: {why}. No start_time can satisfy it, so changing the "
                    "time will not help. Do not call schedule_workflow again for this request; call finish and "
                    "explain the conflict to the user."
                )
                return _response(409, "REQUEST_CONTRADICTION", "TEMPORAL_CONTRADICTION_IN_REQUEST", detail)
            detail = "I cannot schedule this as stated -- the timing is inconsistent: " + self.result.explanation()
            return _response(409, "INFEASIBLE_SCHEDULE", "TEMPORAL_CONTRADICTION", detail)

        steps = args.get("steps") if isinstance(args, dict) else None
        start_time = args.get("start_time") if isinstance(args, dict) else None
        if not isinstance(steps, list) or not isinstance(start_time, str):
            return self._commit(args, real_tool)  # malformed call: SimuHome's own validation answers it
        try:
            start_min = clock_to_minutes(self.base_time, start_time)
        except Exception:
            return self._commit(args, real_tool)

        devices = devices_in_steps(steps)
        candidates = [g for g in self.ir.goals if any(t.device_id in devices for t in g.targets)]
        if not candidates:
            self.stats.unmatched_calls += 1

        # C1: start time against the goal windows
        matched = candidates
        if candidates and self.windows:
            inside = [g for g in candidates if self.windows.get(g.goal_id) is None
                      or self.windows[g.goal_id].contains(start_min, self.cfg.slack_minutes)]
            if inside:
                matched = inside
            elif self.stats.time_rejections < self.cfg.max_plan_rejections:
                self.stats.time_rejections += 1
                return _response(409, "TIME_MISMATCH", "START_TIME_DOES_NOT_MATCH_REQUEST",
                                 self._time_mismatch_detail(start_time, candidates))
            else:
                self.stats.fail_open += 1

        # C2: dry run on a copy of the home
        notes = []
        if start_min > self.cfg.dry_run_horizon_minutes and self.cfg.dry_run != "off":
            self.stats.events.append(("dry_run_skipped_beyond_horizon", start_time))
        elif self.cfg.dry_run != "off" and self.dry_run_fn is not None:
            expected = self._expected_targets(self._closest_goals(matched, start_min), devices) \
                if self.cfg.target_check != "off" else []
            dr = self._dry_run(start_time, steps, expected)
            if dr is not None:
                if dr.status == "failed":
                    if self.cfg.dry_run == "reject" and self.stats.dry_run_rejections < self.cfg.max_plan_rejections:
                        self.stats.dry_run_rejections += 1
                        return _response(409, "DRY_RUN_FAILED", "WORKFLOW_STEP_WOULD_FAIL", (
                            f"Scheduling refused: when this workflow runs at {start_time}, a step would fail: "
                            f"{dr.error}. Check the device's commands and attributes with get_device_structure, "
                            "fix the step, and schedule again."
                        ))
                    notes.append(f"Dry run: a step will fail at {start_time}: {dr.error}.")
                if dr.target_mismatches:
                    msg = "; ".join(f"{m.device_id} {m.attribute} would be {m.actual!r}, the request asks for {m.expected!r}"
                                    for m in dr.target_mismatches)
                    if self.cfg.target_check == "reject" and self.stats.target_rejections < self.cfg.max_plan_rejections:
                        self.stats.target_rejections += 1
                        return _response(409, "TARGET_MISMATCH", "WORKFLOW_WOULD_NOT_REACH_REQUESTED_STATE",
                                         f"Scheduling refused: after this workflow runs at {start_time}, {msg}. "
                                         "Fix the steps and schedule again.")
                    if self.cfg.target_check != "off":
                        self.stats.target_warnings += 1
                        notes.append(f"Dry run after {start_time}: {msg}.")
        resp = self._commit(args, real_tool)
        if notes and _ok(resp) and isinstance(resp.get("data"), dict):
            resp = copy.deepcopy(resp)
            resp["data"]["verifier_note"] = " ".join(notes)
        return resp

    def _closest_goals(self, goals, start_min: float):
        """Per device, keep only the goal whose computed time is closest to start_min.
        (Two goals on one device, e.g. dimmer to 30% at +13 min and to 20% at +21 min,
        must not both be expected from a single call.)"""
        resolved = {g.goal_id: g.resolved_minutes for g in self.result.goal_results}
        best: dict[str, Any] = {}
        for g in goals:
            dist = abs(resolved[g.goal_id] - start_min) if resolved.get(g.goal_id) is not None else float("inf")
            for t in g.targets:
                if t.device_id not in best or dist < best[t.device_id][0]:
                    best[t.device_id] = (dist, g)
        return best

    @staticmethod
    def _expected_targets(best: dict, devices: list[str]) -> list[tuple[str, str, Any]]:
        out = []
        for dev in devices:
            if dev in best:
                g = best[dev][1]
                out.extend((t.device_id, a.attribute, a.value) for t in g.targets if t.device_id == dev for a in t.asserts)
        return out

    def _dry_run(self, start_time: str, steps: list, expected: list):
        t0 = time.time()
        try:
            snapshot = _tool_data(self.call_tool("get_home_state", {}))
            if not isinstance(snapshot, dict):
                raise RuntimeError("no home snapshot")
            dr = self.dry_run_fn(snapshot, {"start_time": start_time, "steps": steps}, self._pending_committed(), expected)
            if dr.status == "error":
                self.stats.events.append(("dry_run_error", (dr.error or "")[:200]))
                self.stats.fail_open += 1
                return None
            return dr
        except Exception as e:
            self.stats.events.append(("dry_run_error", repr(e)[:200]))
            self.stats.fail_open += 1
            return None
        finally:
            self.stats.dry_run_seconds += time.time() - t0

    def _pending_committed(self) -> list[dict]:
        """Workflows this episode scheduled that are still pending in the real home. Ones the
        agent cancelled (cancel_workflow) or that already ran (their effect is in the snapshot)
        must not be replayed on the copy."""
        out = []
        for wf in self.committed:
            wid = wf.get("workflow_id")
            if wid is None:
                out.append(wf)
                continue
            status = (_tool_data(self.call_tool("get_workflow_status", {"workflow_id": wid})) or {}).get("status")
            if status in (None, "pending"):
                out.append(wf)
        return out

    def _time_mismatch_detail(self, start_time: str, goals) -> str:
        parts = []
        for g in goals:
            w = self.windows.get(g.goal_id)
            if w is None:
                continue
            lo = minutes_to_clock(self.base_time, w.lo) if w.lo is not None else None
            hi = minutes_to_clock(self.base_time, w.hi) if w.hi is not None else None
            devs = ", ".join(t.device_id for t in g.targets)
            if lo and hi:
                rng = f"between {lo} and {hi}"
            elif hi:
                rng = f"no later than {hi}"
            else:
                rng = f"no earlier than {lo}"
            parts.append(f"for {devs} the request works out to {rng}")
        return (
            f"Scheduling refused: start_time {start_time} does not match the request ({'; '.join(parts)}). "
            "Recompute the time from the request and the current device states, then schedule again."
        )

    def _commit(self, args: dict, real_tool: Callable[[dict], dict]) -> dict:
        resp = real_tool(args)
        if _ok(resp):
            self.stats.passed += 1
            if isinstance(args, dict) and isinstance(args.get("steps"), list) and isinstance(args.get("start_time"), str):
                wid = (resp.get("data") or {}).get("workflow_id") if isinstance(resp.get("data"), dict) else None
                self.committed.append({"start_time": args["start_time"], "steps": args["steps"], "workflow_id": wid})
        return resp


In [ ]:
%%writefile irlib/shadow_sim.py
"""Component 4: dry run of a proposed workflow on a copy of the home.

SimuHome's `schedule_workflow` only checks the tool name and that the start time
is in the future. Whether the steps can actually execute (right device, command,
cluster, argument names, writable attribute) is discovered only when the workflow
fires, long after the agent has called `finish`. This module finds out at
scheduling time:

    live snapshot (GET /home/state)
      -> SimulationConfig with base_time = the snapshot's current time
      -> a fresh, thread-less Home (SimuHome's own initializer)
      -> re-schedule the workflows already committed in this episode
      -> schedule the candidate, fast-forward to its due tick (+1)
      -> read the workflow status and the target attributes

Everything goes through SimuHome's own code (`_initialize_home_locally`,
`_convert_snapshot_to_simulation_config`, `Home._fast_forward_to`), so nothing in
SimuHome is edited (its licence is CC BY-NC-ND 4.0). This is a lightweight form of
the "simulation-based pre-validation" the SimuHome paper lists as future work.

Imports SimuHome lazily, so the rest of the package works without it.
"""

from __future__ import annotations

import time
from dataclasses import dataclass, field
from typing import Any, Optional


@dataclass
class TargetMismatch:
    device_id: str
    attribute: str
    expected: Any
    actual: Any


@dataclass
class DryRunResult:
    ok: bool  # workflow completed in the dry run
    status: str  # completed | failed | pending | not_scheduled | error
    error: Optional[str] = None
    target_mismatches: list[TargetMismatch] = field(default_factory=list)
    seconds: float = 0.0


def _split_attribute(path: str) -> tuple[int, str, str]:
    endpoint, cluster, attribute = path.split(".", 2)
    return int(endpoint), cluster, attribute


def values_match(expected: Any, actual: Any) -> bool:
    """Plain equality, the same rule SimuHome's QT4 scorer uses (`_verify_asserts`).
    Values are raw attribute values (e.g. LevelControl.CurrentLevel 30), not percentages."""
    return actual == expected


def build_shadow_home(snapshot: dict):
    """A thread-less SimuHome Home whose tick 0 is the snapshot's current time."""
    from src.pipelines.episode_generation.core.home_augmenter import (  # SimuHome's own helpers
        _convert_snapshot_to_simulation_config,
        _initialize_home_locally,
    )

    config = _convert_snapshot_to_simulation_config(snapshot, base_time=snapshot["current_time"])
    home, _ = _initialize_home_locally(config)
    return home


def dry_run(
    snapshot: dict,
    candidate: dict,
    committed: Optional[list[dict]] = None,
    expected_targets: Optional[list[tuple[str, str, Any]]] = None,
) -> DryRunResult:
    """Run `candidate` = {"start_time", "steps"} on a copy of the home.

    `committed`: workflows the agent already scheduled in this episode (same
    shape); replayed first so the copy sees their effects. `expected_targets`:
    (device_id, "1.Cluster.Attribute", value) triples to read after the candidate
    ran; mismatches are reported, never raised.
    """
    t0 = time.time()
    try:
        home = build_shadow_home(snapshot)
    except Exception as e:  # snapshot the initializer cannot rebuild: say so, don't guess
        return DryRunResult(ok=False, status="error", error=f"shadow build failed: {e!r}"[:300], seconds=time.time() - t0)

    for wf in committed or []:
        home.schedule_workflow(wf["start_time"], wf["steps"])  # ones already in the past are skipped by Home

    res = home.schedule_workflow(candidate["start_time"], candidate["steps"])
    if not res.success:
        return DryRunResult(ok=False, status="not_scheduled", error=res.error_message, seconds=time.time() - t0)
    wf_id = res.data["workflow_id"]
    wf = home.workflows_by_id[wf_id]

    ff = home._fast_forward_to(int(wf["due_tick"]) + 1)  # noqa: SLF001 (SimuHome's own fast-forward)
    if not ff.success:
        return DryRunResult(ok=False, status="error", error=f"fast-forward failed: {ff.error_message}", seconds=time.time() - t0)

    mismatches = []
    for device_id, attr_path, expected in expected_targets or []:
        try:
            endpoint, cluster, attribute = _split_attribute(attr_path)
            got = home._get_attribute(device_id, endpoint, cluster, attribute)  # noqa: SLF001
            actual = got.data if got.success else None
        except Exception:
            actual = None
        if not values_match(expected, actual):
            mismatches.append(TargetMismatch(device_id, attr_path, expected, actual))

    status = wf.get("status", "pending")
    return DryRunResult(
        ok=status == "completed",
        status=status,
        error=wf.get("error"),
        target_mismatches=mismatches,
        seconds=time.time() - t0,
    )


In [ ]:
%%writefile irlib/harness.py
"""Wires the approach-2 gate into SimuHome's ReAct agent, one gate per episode.

Two monkeypatches (SimuHome's source is CC BY-NC-ND 4.0, so it is never edited):
  - `ReActAgent.run`: builds the episode's EpisodeGate (parse once, pre-flight once)
    and, depending on the arm, appends a hint to the query, answers the user
    without calling the agent, or appends the prompt-only self-check instruction.
  - `TOOL_REGISTRY["schedule_workflow"]`: routes each call to the gate of the
    episode running in the CURRENT THREAD (several episodes run in parallel).

The gate's own tool calls (current time, CountdownTime, home snapshot) go through
SimuHome's `run_tool` in the same thread, so they reach that episode's simulator,
and they are not recorded as agent actions (scoring is unaffected).

Imports SimuHome lazily (call `install()` after SimuHome is on sys.path).
"""

from __future__ import annotations

import threading
import time
from typing import Any, Callable, Optional

from .gate import ARMS, EpisodeGate, GateConfig

SELF_CHECK_PROMPT = (
    " IMPORTANT: before scheduling anything, work out the exact time of every requested action. If two parts of the"
    " request give the same event two different times (a contradiction), do NOT call schedule_workflow; call finish"
    " and explain the conflict."
)


class Harness:
    def __init__(
        self,
        parse_fn: Callable[[str], dict],
        dry_run_fn: Optional[Callable[..., Any]] = None,
        ir_version: int = 2,
        arms: Optional[dict[str, Optional[GateConfig]]] = None,
    ):
        self.parse_fn = parse_fn
        self.dry_run_fn = dry_run_fn
        self.ir_version = ir_version
        self.arms = dict(arms or ARMS)
        self.arm = "baseline"
        self._tls = threading.local()
        self._lock = threading.Lock()
        self.telemetry: list[dict] = []
        self._installed = False

    # ------------------------------------------------------------ public API
    def set_arm(self, arm: str) -> None:
        if arm not in self.arms:
            raise KeyError(f"unknown arm {arm!r}; known: {sorted(self.arms)}")
        self.arm = arm

    def begin_episode(self, key: str) -> None:
        """Called by the runner in the worker thread before each episode (labels telemetry)."""
        self._tls.episode = key

    def install(self) -> None:
        if self._installed:
            return
        from src.agents.strategies.base import AgentResult, AgentStep
        from src.agents.strategies.react_agent import ReActAgent
        from src.agents.tools import TOOL_REGISTRY, run_tool, tool_schedule_workflow

        self._AgentResult, self._AgentStep, self._run_tool = AgentResult, AgentStep, run_tool
        original_tool = tool_schedule_workflow
        original_run = getattr(ReActAgent.run, "_approach2_orig", ReActAgent.run)  # idempotent if re-installed
        harness = self

        def gated_schedule_workflow(args):
            gate = getattr(harness._tls, "gate", None)
            if gate is None:
                return original_tool(args)
            return gate.on_schedule(args, original_tool)

        # The agent's prompt lists each tool from its docstring; without this the agent
        # never sees schedule_workflow's argument description.
        gated_schedule_workflow.__doc__ = original_tool.__doc__
        TOOL_REGISTRY["schedule_workflow"] = gated_schedule_workflow

        def patched_run(agent_self, user_query, **kwargs):
            return harness._run_episode(original_run, agent_self, user_query, **kwargs)

        patched_run._approach2_orig = original_run
        ReActAgent.run = patched_run
        self._installed = True

    # --------------------------------------------------------------- internals
    def _parse(self, query: str) -> tuple[Optional[dict], float, Optional[str]]:
        t0 = time.time()
        try:
            return self.parse_fn(query), time.time() - t0, None
        except Exception as e:  # parse failure -> the gate fails open
            return None, time.time() - t0, repr(e)[:200]

    def _run_episode(self, original_run, agent_self, user_query: str, **kwargs):
        arm = self.arm
        cfg = self.arms[arm]
        query = user_query
        gate = None
        record = {"episode": getattr(self._tls, "episode", None), "arm": arm, "query": user_query[:200],
                  "answered_by_gate": False, "parse_seconds": 0.0, "parse_error": None}
        t0 = time.time()
        try:
            if cfg is not None:
                cfg = GateConfig(**{**cfg.__dict__, "ir_version": self.ir_version})
                compact, record["parse_seconds"], record["parse_error"] = self._parse(user_query)
                base_time = kwargs.get("current_time")
                if not base_time or base_time == "None":
                    base_time = (self._run_tool("get_current_time", {}).get("data") or {}).get("now")
                gate = EpisodeGate(user_query, base_time, compact, cfg, self._run_tool, self.dry_run_fn)
                self._tls.gate = gate
                action, text = gate.preflight_action()
                if action == "answer":
                    record["answered_by_gate"] = True
                    step = self._AgentStep(step=1, thought="Answered by the scheduling verifier before the agent ran.",
                                           action="finish", action_input={"answer": text}, observation=None)
                    return self._AgentResult(steps=[step], tool_calls=[], final_answer=text, raw_responses=[], events=[])
                if action == "hint":
                    query = query + text
            if arm == "selfcheck":
                query = query + SELF_CHECK_PROMPT
            return original_run(agent_self, query, **kwargs)
        finally:
            self._tls.gate = None
            record["episode_seconds"] = round(time.time() - t0, 2)
            if gate is not None:
                record.update(gate.stats.as_dict())
            with self._lock:
                self.telemetry.append(record)


In [ ]:
%%writefile irlib/parallel_runner.py
"""Runs SimuHome episodes several at a time, in this process.

Approach 1 ran one episode at a time (SimuHome's evaluator runs a model's episodes
sequentially, one simulator per model). With ~45 s per hosted-LLM call that used
under 2 of the 40 requests/minute a free NVIDIA key allows. Here each worker
thread owns one simulator server (its own port) and runs SimuHome's own
`run_single_config` (the same agent, scorer and judge panel). All LLM calls
still pass through the notebook's request shim, whose global throttle keeps the
total under the key's rate limit however many threads run.

Results are written exactly where and how SimuHome's evaluator writes them
(`<output_root>/<run_id>/<safe model name>/<episode file name>`, same record and
error-record format), so SimuHome's aggregator reads them unchanged.

Imports SimuHome lazily (call after SimuHome is on sys.path, with cwd = SimuHome).
"""

from __future__ import annotations

import json
import os
import queue
import socket
import subprocess
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from typing import Callable, Optional


def _free_port() -> int:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return int(s.getsockname()[1])


def start_simulators(n: int, simuhome_dir: str = ".", wait_seconds: float = 60.0, log_dir: Optional[str] = None):
    """Start n SimuHome simulator servers; returns [(port, Popen)]."""
    import requests

    servers = []
    for i in range(n):
        port = _free_port()
        env = dict(os.environ, SERVER_PORT=str(port))
        log = open(os.path.join(log_dir, f"sim_{port}.log"), "w") if log_dir else subprocess.DEVNULL
        proc = subprocess.Popen([sys.executable, "-m", "src.simulator.api.app"], cwd=simuhome_dir, env=env,
                                stdout=log, stderr=subprocess.STDOUT)
        servers.append((port, proc))
    deadline = time.time() + wait_seconds
    for port, proc in servers:
        while True:
            if proc.poll() is not None:
                raise RuntimeError(f"simulator on port {port} exited early (code {proc.returncode})")
            try:
                if requests.get(f"http://127.0.0.1:{port}/api/__health__", timeout=1).ok:
                    break
            except Exception:
                pass
            if time.time() > deadline:
                raise RuntimeError(f"simulator on port {port} did not come up in {wait_seconds:.0f} s")
            time.sleep(0.5)
    return servers


def stop_simulators(servers) -> None:
    for _, proc in servers:
        if proc.poll() is None:
            proc.terminate()
    for _, proc in servers:
        try:
            proc.wait(timeout=5)
        except Exception:
            proc.kill()


def episode_paths(benchmark_dir: str, qt: str, case: str, seeds) -> list[Path]:
    out = []
    for s in seeds:
        p = Path(benchmark_dir) / f"{qt}_{case}_seed_{s}.json"
        if p.exists():
            out.append(p)
    return out


def run_episodes(
    cfg_paths: list[Path],
    *,
    servers,
    run_id: str,
    model: str,
    make_main_llm: Callable[[], object],
    make_judge_llms: Callable[[], list],
    output_root: str = "experiments",
    max_steps: int = 20,
    timeout: float = 120.0,
    on_episode_start: Optional[Callable[[str], None]] = None,
    skip_done: bool = True,
    log: Callable[[str], None] = print,
) -> dict:
    """Run every episode in `cfg_paths` with len(servers) workers. Returns counts."""
    from src.cli.episode_evaluator import (
        _build_error_record,
        _classify_runtime_failure,
        _is_valid_existing_result,
        _output_path_for_config,
        _write_json_atomic,
    )
    from src.pipelines.episode_evaluation.runner import run_single_config

    ports: "queue.Queue[int]" = queue.Queue()
    for port, _ in servers:
        ports.put(port)
    counts = {"total": len(cfg_paths), "skipped": 0, "success": 0, "failed": 0}
    lock = threading.Lock()
    t_start = time.time()

    def one(cfg_path: Path) -> None:
        out_path = _output_path_for_config(cfg_path, output_root, run_id, model)
        if skip_done and _is_valid_existing_result(out_path):
            with lock:
                counts["skipped"] += 1
            return
        port = ports.get()
        try:
            if on_episode_start:
                on_episode_start(f"{run_id}/{cfg_path.stem}")
            t0 = time.time()
            try:
                record = run_single_config(
                    cfg_path=str(cfg_path), base_url=f"http://127.0.0.1:{port}/api", timeout=float(timeout),
                    max_steps=int(max_steps), agent_strategy="react",
                    main_llm=make_main_llm(), judge_llms=make_judge_llms(),
                )
                _write_json_atomic(out_path, record)
                score = (record.get("evaluation_result") or {}).get("score")
                with lock:
                    counts["success"] += 1
                log(f"  {cfg_path.stem}: score={score} ({time.time() - t0:.0f} s)")
            except Exception as exc:
                score, error_type, category, attempted = _classify_runtime_failure(exc)
                _write_json_atomic(out_path, _build_error_record(
                    cfg_path=cfg_path, model=model, strategy="react", error_type=error_type, message=str(exc),
                    score=score, error_category=category, attempted=attempted))
                with lock:
                    counts["failed"] += 1
                log(f"  {cfg_path.stem}: FAILED {error_type}: {str(exc)[:160]}")
        finally:
            ports.put(port)

    with ThreadPoolExecutor(max_workers=max(1, len(servers))) as pool:
        list(pool.map(one, cfg_paths))

    counts["seconds"] = round(time.time() - t_start, 1)
    summary_path = Path(output_root) / run_id / "run_summary.json"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(json.dumps({"run_id": run_id, "model": model, "workers": len(servers), **counts}, indent=2))
    return counts


## 2. Sanity checks (seconds, no API)

Joint Z3 encoder on the ground-truth IR: all benchmark verdicts, and every benchmark goal time inside its
window (the same checks as `tests/test_pipeline.py`, which pass locally: 286/286 and 301/301).


In [ ]:
import glob, json
from irlib.extract_ir_from_episode import extract_episode
from irlib.constraint_encoder import check_feasibility, goal_windows
BENCH = os.path.join(PROJECT_ROOT, 'SimuHome/data/benchmark')
ok = n = inside = goals = 0
for path in sorted(glob.glob(f'{BENCH}/qt4-*.json')):
    ep = json.load(open(path, encoding='utf-8'))
    if ep['meta'].get('conflict_type') == 'completion_vs_pause':
        continue
    ir = extract_episode(ep, for_training=False)
    n += 1; ok += check_feasibility(ir).feasible == (ep['meta']['case'] == 'feasible')
    if ep['meta']['case'] == 'feasible':
        w = goal_windows(ir); tick = float(ep['initial_home_config']['tick_interval'])
        for gi, g in enumerate(ep['eval']['goals']):
            goals += 1; inside += w[gi].contains(g['when']['at_tick'] * tick / 60.0, 1e-6)
print(f'verdicts {ok}/{n}, goal times inside their window {inside}/{goals}')
assert n == 286 and goals == 301 and ok == n and inside == goals, (n, ok, goals, inside)


## 3. Agent LLM backend and request shim

The agent is a hosted model behind an OpenAI-compatible endpoint (default: `google/gemma-4-31b-it` on
NVIDIA's free API, 40 requests/min per key). The shim patches the OpenAI SDK once in this process: it spaces
calls out with ONE global throttle (so parallel episodes together stay under the key's limit), counts calls,
retries provider errors with backoff, and drops parameters NVIDIA rejects.

**Several notebooks on one key:** set `NOTEBOOKS_SHARING_KEY` to how many run at the same time (for
example 3 notebooks, one agent model each); each then takes 1/3 of the budget.


In [ ]:
import time, threading, copy
# ============================ EDIT THIS BLOCK ONLY ============================
LLM_API_BASE = 'https://integrate.api.nvidia.com/v1'
LLM_MODEL = 'google/gemma-4-31b-it'      # the agent; change per notebook for a multi-model study
JUDGE_MODEL = 'google/gemma-4-31b-it'    # scores infeasible episodes (3 calls each); keep fixed across arms
LLM_API_KEY_ENV = 'NVIDIA_API_KEY'       # Kaggle secret name
KEY_RPM = 40                             # the key's requests-per-minute limit
NOTEBOOKS_SHARING_KEY = 1                # notebooks using this key at the same time
LLM_CALL_CAP = 4000                      # hard stop for this session (agent + judge calls, retries included)
# ==============================================================================
try:
    from kaggle_secrets import UserSecretsClient
    os.environ[LLM_API_KEY_ENV] = UserSecretsClient().get_secret(LLM_API_KEY_ENV)
except Exception as e:
    print('Kaggle secret not read (set the env var yourself if not on Kaggle):', repr(e)[:120])
MIN_SECONDS_BETWEEN_CALLS = 60.0 / (0.92 * KEY_RPM / NOTEBOOKS_SHARING_KEY)   # 8% headroom
DROP_PARAMS = ['seed', 'response_format']   # NVIDIA lists structured output as unsupported; JSON is parsed from text

import openai
from openai.resources.chat.completions import Completions
SHIM_RETRIES = 4
SHIM_ERRORS = {}   # provider errors seen so far, by type (timeouts, 429s, 5xx), for the run log
_RETRYABLE = (openai.InternalServerError, openai.RateLimitError, openai.APITimeoutError, openai.APIConnectionError)
LLM_CALLS = {'count': 0, 'last': 0.0}
_llm_lock = threading.Lock()
_orig_create = getattr(Completions.create, '_orig', Completions.create)
def _shimmed_create(self, *args, **kwargs):
    for k in DROP_PARAMS:
        kwargs.pop(k, None)
    for attempt in range(SHIM_RETRIES + 1):
        with _llm_lock:   # one throttle for every thread in this process
            if LLM_CALL_CAP and LLM_CALLS['count'] >= LLM_CALL_CAP:
                raise RuntimeError(f'LLM call cap reached ({LLM_CALL_CAP})')
            wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - LLM_CALLS['last'])
            if wait > 0:
                time.sleep(wait)
            LLM_CALLS['last'] = time.time(); LLM_CALLS['count'] += 1
        try:
            return _orig_create(self, *args, **kwargs)
        except _RETRYABLE as e:
            if attempt == SHIM_RETRIES:
                raise
            delay = 30 * 2 ** attempt        # 30, 60, 120, 240 s
            SHIM_ERRORS[type(e).__name__] = SHIM_ERRORS.get(type(e).__name__, 0) + 1
            print(f'[shim] {type(e).__name__}: retry {attempt + 1}/{SHIM_RETRIES} in {delay}s '
                  f'(this is a notice; the run continues. totals so far: {SHIM_ERRORS})', flush=True)
            time.sleep(delay)
_shimmed_create._orig = _orig_create
Completions.create = _shimmed_create
print(f'shim installed: at most one call every {MIN_SECONDS_BETWEEN_CALLS:.2f} s from this notebook')

from openai import OpenAI
r = OpenAI(api_key=os.environ.get(LLM_API_KEY_ENV, ''), base_url=LLM_API_BASE).chat.completions.create(
    model=LLM_MODEL, temperature=0.0, messages=[{'role': 'user', 'content': 'Reply with the word OK.'}])
print('LLM preflight OK:', (r.choices[0].message.content or '')[:60])


## 4. The parser: cache first, GPU only if needed

Looks for `parse_cache_v3.json` (then `parse_cache_v2.json`) in `/kaggle/input/*`. Without one, it loads the
adapter on a GPU, parses every held-out request once, and saves the cache to `/kaggle/working` so later
sessions can run on CPU. `IR_VERSION` decides whether the start-time check (C1) is used: only v3 IRs know
"after the previous action", so with a v2 cache C1 is switched off automatically.


In [ ]:
# ============================ EDIT ONLY IF NEEDED ============================
PARSE_CACHE_PATH = None   # e.g. '/kaggle/input/my-dataset/parser_v3_outputs_new/parse_cache_v3.json'; None = search
# ==============================================================================
def find_all(name):
    # any depth under /kaggle/input (Kaggle nests uploaded folders), plus the working directory
    return sorted(set(glob.glob(f'/kaggle/input/**/{name}', recursive=True) + glob.glob(f'**/{name}', recursive=True)))

def find_input(name):
    hits = find_all(name)
    return hits[0] if hits else None

def _cache_meta(path):
    try:
        return json.load(open(path, encoding='utf-8')).get('_meta', {})
    except Exception:
        return {}

PARSE_CACHE, IR_VERSION, CACHE_META = {}, 2, {}
candidates = [PARSE_CACHE_PATH] if PARSE_CACHE_PATH else find_all('parse_cache_v3.json') + find_all('parse_cache_v2.json')
print('parse caches found:', candidates or 'NONE')
if len(candidates) > 1 and not PARSE_CACHE_PATH:
    # several attached (e.g. old and new parser outputs): take the newest v3 by its creation stamp, and say so
    candidates.sort(key=lambda p: (str(_cache_meta(p).get('ir_version', 0)), 'run_tag' in _cache_meta(p),
                                   str(_cache_meta(p).get('created', ''))), reverse=True)
    print('WARNING: several parse caches attached; using the newest:', candidates[0])
if candidates:
    path = candidates[0]
    raw = json.load(open(path, encoding='utf-8'))
    PARSE_CACHE = {k: v for k, v in raw.items() if k != '_meta'}
    CACHE_META = raw.get('_meta', {})
    IR_VERSION = int(CACHE_META.get('ir_version', 3 if 'v3' in os.path.basename(path) else 2))
    print(f'parse cache: {path} ({len(PARSE_CACHE)} requests, IR v{IR_VERSION})')
    print('   made by:', CACHE_META.get('run_tag', 'unknown run'), '| adapter sha256', CACHE_META.get('adapter_sha256', '?'))

live_parse = None
if not PARSE_CACHE:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel
    SYSTEM_PROMPT = (
        'You convert a smart-home scheduling request into a strict JSON '
        'intermediate representation (IR). Output ONLY the JSON object, '
        'no prose, no code fences.'
    )
    adapter = None
    for version, name in ((3, 'qwen3_ir_parser_adapter_v3'), (2, 'qwen3_ir_parser_adapter_v2')):
        adapter = find_input(name)
        if adapter:
            IR_VERSION = version; break
    assert adapter and torch.cuda.is_available(), 'no parse cache and no adapter+GPU: attach parse_cache_v3.json (or an adapter on a GPU session)'
    tok = AutoTokenizer.from_pretrained(adapter)
    _dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float32   # T4: no bf16; fp32 is safest
    base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-1.7B', torch_dtype=_dtype, device_map={'': 0})
    parser_model = PeftModel.from_pretrained(base, adapter).eval()
    _gpu_lock = threading.Lock()
    def live_parse(query):
        prompt = f"<|system|>\n{SYSTEM_PROMPT}\n<|user|>\n{query}\n<|assistant|>\n"
        with _gpu_lock, torch.no_grad():
            ids = tok(prompt, return_tensors='pt').to(parser_model.device)
            out = parser_model.generate(**ids, max_new_tokens=700, do_sample=False)
        return json.loads(tok.decode(out[0][ids['input_ids'].shape[1]:], skip_special_tokens=True))
    from irlib.extract_ir_from_episode import extract_dataset
    for row in extract_dataset(BENCH):
        if int(row['meta']['seed']) % 5 == 0:   # the held-out split
            t0 = time.time()
            try:
                PARSE_CACHE[row['query']] = {'ir': live_parse(row['query']), 'seconds': round(time.time() - t0, 3)}
            except Exception as e:
                PARSE_CACHE[row['query']] = {'error': repr(e)[:200], 'seconds': round(time.time() - t0, 3)}
    json.dump({'_meta': {'ir_version': IR_VERSION, 'adapter': adapter}, **PARSE_CACHE},
              open(os.path.join(WORK, f'parse_cache_v{IR_VERSION}.json'), 'w'), ensure_ascii=False)
    print(f'built parse_cache_v{IR_VERSION}.json for {len(PARSE_CACHE)} held-out requests')

def parse_fn(query):
    entry = PARSE_CACHE.get(query)
    if entry is None and live_parse is not None:
        return live_parse(query)
    if entry is None:
        raise KeyError('request not in the parse cache')
    if 'error' in entry:
        raise ValueError(entry['error'])
    return copy.deepcopy(entry['ir'])
print('IR version', IR_VERSION, '-> start-time check (C1)', 'ON' if IR_VERSION >= 3 else 'OFF (v2 IRs cannot express goal chains)')


## 5. Install the gate

`Harness.install()` patches SimuHome's `ReActAgent.run` and `TOOL_REGISTRY['schedule_workflow']` in this
process (no SimuHome file is edited). One gate per episode, kept per thread, so episodes can run in parallel.

| arm | pre-flight | in `schedule_workflow` |
|---|---|---|
| `baseline` | - | SimuHome's tool, unchanged |
| `selfcheck` | prompt-only instruction to check the times | unchanged |
| `v1_gate` | - | approach 1: request-level verdict, plain 409 |
| `pre_hint` | contradiction -> note to the agent at step 0 | final rejection |
| `pre_answer` | contradiction -> the gate answers, agent not called | final rejection |
| `full` | as `pre_hint` | + start-time window (C1, v3 only) + dry run (C2) |
| `full_answer` | as `pre_answer` | + C1 + C2 |


In [ ]:
os.chdir(os.path.join(PROJECT_ROOT, 'SimuHome'))   # SimuHome resolves its data and prompts from here
from irlib.harness import Harness
from irlib.gate import ARMS
from irlib.shadow_sim import dry_run
harness = Harness(parse_fn=parse_fn, dry_run_fn=dry_run, ir_version=IR_VERSION)
harness.install()
for name, cfg in ARMS.items():
    print(f'{name:12s}', cfg if cfg is None else {k: v for k, v in cfg.__dict__.items() if k in ('preflight', 'terminal_rejection', 'time_check', 'dry_run', 'target_check')})


## 6. E1 plan runner

Edit only the `PLAN` block. Each entry `(sub-type, case, arm)` runs on the held-out seeds (seed % 5 == 0,
the episodes the parser never trained on). `WORKERS` episodes run at once, each with its own simulator.
Finished runs are skipped on a re-run; a run is redone if it had API errors; no run starts past
`BUDGET_HOURS`; two bad runs in a row stop the plan. Everything is saved after every run.

Rough cost: an episode is 2-10 agent calls (+3 judge calls if infeasible). At ~45 s per call and 6 workers,
10 episodes take ~10-15 minutes instead of ~1 hour.


In [ ]:
import collections, shutil, subprocess
from pathlib import Path
from irlib.parallel_runner import start_simulators, stop_simulators, episode_paths, run_episodes
from src.agents.providers.openai_provider import OpenAIChatProvider

# ============================ EDIT THIS BLOCK ONLY ============================
WORKERS = 3            # episodes at once (each gets its own simulator). The free NVIDIA endpoint times out more
                       # when many long requests are in flight; raise to 6 only if no [shim] timeouts appear
REQUEST_TIMEOUT = 300  # seconds one LLM request may take before it counts as timed out
E1_MAX_SEEDS = 10      # episodes per (sub-type, case) per arm; the held-out split has 9-10 per cell
BUDGET_HOURS = 8.0
PLAN = [               # (sub-type, case, arm), in priority order
    ('qt4-1', 'infeasible', 'v1_gate'), ('qt4-1', 'infeasible', 'pre_hint'), ('qt4-1', 'infeasible', 'full_answer'),
    ('qt4-1', 'feasible',   'v1_gate'), ('qt4-1', 'feasible',   'full'),
    ('qt4-2', 'infeasible', 'baseline'), ('qt4-2', 'infeasible', 'selfcheck'), ('qt4-2', 'infeasible', 'pre_hint'), ('qt4-2', 'infeasible', 'full_answer'),
    ('qt4-2', 'feasible',   'baseline'), ('qt4-2', 'feasible',   'full'),
    ('qt4-3', 'infeasible', 'baseline'), ('qt4-3', 'infeasible', 'selfcheck'), ('qt4-3', 'infeasible', 'pre_hint'), ('qt4-3', 'infeasible', 'full_answer'),
    ('qt4-3', 'feasible',   'baseline'), ('qt4-3', 'feasible',   'full'),
    ('qt4-1', 'infeasible', 'baseline'), ('qt4-1', 'infeasible', 'selfcheck'), ('qt4-1', 'feasible', 'baseline'),
]
# ==============================================================================
SAFE_MODEL = LLM_MODEL.replace('/', '_').replace(':', '_')
api_key = os.environ.get(LLM_API_KEY_ENV, '')
def make_provider(model, seed):
    # ONE retry layer only: the request shim (section 3). Otherwise the OpenAI library (2 retries) and SimuHome's
    # provider (10 retries) stack on top of the shim's 4, and one dead request can stall an episode for over an hour.
    p = OpenAIChatProvider(model=model, temperature=0.0, seed=seed, api_key=api_key, api_base=LLM_API_BASE,
                           timeout=REQUEST_TIMEOUT, max_retries=0)
    p._client = p._client.with_options(max_retries=0, timeout=REQUEST_TIMEOUT)
    return p
make_main_llm = lambda: make_provider(LLM_MODEL, 42)
make_judges = lambda: [make_provider(JUDGE_MODEL, 42 + i) for i in range(3)]

def held_out_seeds(qt, case):
    seeds = sorted(int(p.split('_seed_')[1].split('.')[0]) for p in glob.glob(f'{BENCH}/{qt}_{case}_seed_*.json'))
    return [s for s in seeds if s % 5 == 0][:E1_MAX_SEEDS]

def summarize(run_id):
    """Accuracy plus the cost columns approach 2 is about: agent LLM calls and seconds per episode."""
    files = [f for f in glob.glob(f'experiments/{run_id}/{SAFE_MODEL}/*.json') if not f.endswith('aggregate_summary.json')]
    recs = [json.load(open(f, encoding='utf-8')) for f in files]
    ev = [r for r in recs if (r.get('evaluation_result') or {}).get('score') in (0, 1)]
    infra = sum((r.get('evaluation_result') or {}).get('score') == -1 for r in recs)
    n = max(1, len(ev))
    return {'evaluated': len(ev), 'correct': sum(r['evaluation_result']['score'] for r in ev), 'infra_errors': infra,
            'accuracy': round(sum(r['evaluation_result']['score'] for r in ev) / n, 3),
            'agent_calls_per_episode': round(sum(len(r.get('raw_responses') or []) for r in ev) / n, 2),
            'seconds_per_episode': round(sum(r.get('duration', 0) for r in ev) / n, 1),
            'schedule_calls_per_episode': round(sum(sum(1 for s in (r.get('steps') or []) if s.get('action') == 'schedule_workflow') for r in ev) / n, 2)}

agg_file, log_file, tel_file = (os.path.join(PROJECT_ROOT, f) for f in ('e1_summary_v2.json', 'e1_run_log_v2.json', 'gate_telemetry.json'))
summary = json.load(open(agg_file)) if os.path.exists(agg_file) else {}
run_log = json.load(open(log_file)) if os.path.exists(log_file) else []

servers = start_simulators(WORKERS, '.', log_dir=None)
print(f'{len(servers)} simulators up on ports', [p for p, _ in servers])
T0, slowest, bad_streak = time.time(), 0.0, 0
try:
    for qt, case, arm in PLAN:
        run_id = f'a2_{arm}_{qt}_{case}'.replace('-', '_')
        prev = summary.get(f'{arm}|{qt}|{case}|{LLM_MODEL}')
        if prev and prev['evaluated'] > 0 and prev['infra_errors'] == 0:
            print('already complete:', run_id); continue
        if (time.time() - T0 + slowest) / 3600 > BUDGET_HOURS:
            print('time budget reached; stopping before', run_id); break
        if LLM_CALL_CAP and LLM_CALLS['count'] + 200 > LLM_CALL_CAP:
            print('call cap nearly reached; stopping before', run_id); break
        shutil.rmtree(f'experiments/{run_id}', ignore_errors=True)
        harness.set_arm(arm)
        calls0, tel0, t_run = LLM_CALLS['count'], len(harness.telemetry), time.time()
        print(f'== {run_id} ({LLM_MODEL})')
        entry = {'run_id': run_id, 'arm': arm, 'cell': f'{qt}|{case}', 'model': LLM_MODEL, 'workers': WORKERS,
                 'ir_version': IR_VERSION, 'started': time.strftime('%Y-%m-%d %H:%M:%S')}
        try:
            paths = episode_paths(BENCH, qt, case, held_out_seeds(qt, case))
            if not paths:
                raise RuntimeError(f'no episode files for {qt} {case} under {BENCH}')
            run_episodes(paths, servers=servers, run_id=run_id, model=LLM_MODEL, make_main_llm=make_main_llm,
                         make_judge_llms=make_judges, on_episode_start=harness.begin_episode)
            s = summarize(run_id)
            summary[f'{arm}|{qt}|{case}|{LLM_MODEL}'] = s
            entry.update(s)
            print(f"   accuracy {s['correct']}/{s['evaluated']} = {s['accuracy']:.2f} | agent calls/ep {s['agent_calls_per_episode']} | "
                  f"s/ep {s['seconds_per_episode']} | infra errors {s['infra_errors']}")
            bad_streak = bad_streak + 1 if s['infra_errors'] else 0
        except Exception as e:
            print('RUN FAILED:', run_id, repr(e)[:300]); entry['error'] = repr(e)[:300]; bad_streak += 1
        entry.update(seconds=round(time.time() - t_run), llm_calls=LLM_CALLS['count'] - calls0)
        gate_rows = harness.telemetry[tel0:]
        for key in ('unsat_rejections', 'time_rejections', 'dry_run_rejections', 'target_warnings', 'fail_open'):
            entry[f'gate_{key}'] = sum(r.get(key, 0) for r in gate_rows)
        entry['gate_answered'] = sum(r.get('answered_by_gate', False) for r in gate_rows)
        slowest = max(slowest, entry['seconds'])
        run_log.append(entry)
        json.dump(summary, open(agg_file, 'w'), indent=2)
        json.dump(run_log, open(log_file, 'w'), indent=2)
        json.dump(harness.telemetry, open(tel_file, 'w'), indent=2, default=str)
        print(f"   {entry['seconds'] // 60} min | {entry['llm_calls']} LLM calls (agent + judge) | gate: "
              f"{ {k[5:]: v for k, v in entry.items() if k.startswith('gate_')} }")
        if bad_streak >= 2:
            print('two bad runs in a row: stopping. Check the provider message above.'); break
finally:
    stop_simulators(servers)
    harness.set_arm('baseline')
print(f'plan finished after {(time.time() - T0) / 3600:.1f} h; {LLM_CALLS["count"]} LLM calls')


## 7. Results table and download bundle

`e1_summary_v2.json` has one row per (arm, cell, model) with accuracy and the cost columns;
`gate_telemetry.json` has one row per episode (pre-flight verdict, every rejection kind, dry-run time).
The per-episode SimuHome records are under `SimuHome/experiments/a2_*`.


In [ ]:
import csv, zipfile
os.chdir(PROJECT_ROOT)
rows = []
for key, s in json.load(open('e1_summary_v2.json')).items():
    arm, qt, case, model = key.split('|')
    rows.append([qt, case, arm, model, s['correct'], s['evaluated'], s['accuracy'], s['agent_calls_per_episode'],
                 s['seconds_per_episode'], s['schedule_calls_per_episode'], s['infra_errors']])
rows.sort()
print(f"{'sub-type':8s} {'case':10s} {'arm':12s} correct/n  acc   calls/ep  s/ep   sched/ep infra")
for r in rows:
    print(f'{r[0]:8s} {r[1]:10s} {r[2]:12s} {r[4]:>3}/{r[5]:<3}   {r[6]:.2f}  {r[7]:>6}  {r[8]:>6}  {r[9]:>6}  {r[10]:>3}')
with open('e1_summary_v2.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['sub_type', 'case', 'arm', 'model', 'correct', 'evaluated', 'accuracy', 'agent_calls_per_episode',
                'seconds_per_episode', 'schedule_calls_per_episode', 'infra_errors'])
    w.writerows(rows)
OUT = os.path.join(WORK, 'approach2_results.zip')
keep = ['e1_summary_v2.json', 'e1_summary_v2.csv', 'e1_run_log_v2.json', 'gate_telemetry.json'] + glob.glob('parse_cache_v*.json') + glob.glob('SimuHome/experiments/a2_*')
with zipfile.ZipFile(OUT, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in keep:
        if os.path.isfile(p):
            z.write(p)
        elif os.path.isdir(p):
            for root, _, files in os.walk(p):
                for fn in files:
                    z.write(os.path.join(root, fn))
print(OUT, round(os.path.getsize(OUT) / 1e6, 1), 'MB')
